In [ ]:
from pathlib import Path

# ==================================================
# USER CONFIGURATION
# Change values only in this block
# ==================================================

PROJECT_ROOT = Path(
    r"/data/notebooks/SDENetFusion"
)

# --------------------------------------------------
# Dataset paths
# --------------------------------------------------
DATASET_ROOT = PROJECT_ROOT / "Datasets" / "Glas"

TRAIN_IMAGE_DIR = DATASET_ROOT / "train" / "img"
TRAIN_ANNOTATION_DIR = DATASET_ROOT / "train" / "ann"

TEST_A_IMAGE_DIR = DATASET_ROOT / "test_a" / "img"
TEST_A_ANNOTATION_DIR = DATASET_ROOT / "test_a" / "ann"

TEST_B_IMAGE_DIR = DATASET_ROOT / "test_b" / "img"
TEST_B_ANNOTATION_DIR = DATASET_ROOT / "test_b" / "ann"

# --------------------------------------------------
# Precomputed patch paths
# --------------------------------------------------
PRECOMPUTED_ROOT = DATASET_ROOT / "precomputed"

TRAIN_PATCH_DIR = (
    PRECOMPUTED_ROOT
    / "train_256_stride_128"
)

TRAIN_PATCH_IMAGE_DIR = (
    TRAIN_PATCH_DIR
    / "images"
)

TRAIN_PATCH_MASK_DIR = (
    TRAIN_PATCH_DIR
    / "masks"
)

TRAIN_PATCH_MANIFEST = (
    TRAIN_PATCH_DIR
    / "manifest.csv"
)

TEST_A_PATCH_DIR = (
    PRECOMPUTED_ROOT
    / "test_a_256_stride_128"
)

TEST_A_PATCH_IMAGE_DIR = (
    TEST_A_PATCH_DIR
    / "images"
)

TEST_A_PATCH_MASK_DIR = (
    TEST_A_PATCH_DIR
    / "masks"
)

TEST_A_PATCH_MANIFEST = (
    TEST_A_PATCH_DIR
    / "manifest.csv"
)

TEST_B_PATCH_DIR = (
    PRECOMPUTED_ROOT
    / "test_b_256_stride_128"
)

TEST_B_PATCH_IMAGE_DIR = (
    TEST_B_PATCH_DIR
    / "images"
)

TEST_B_PATCH_MASK_DIR = (
    TEST_B_PATCH_DIR
    / "masks"
)

TEST_B_PATCH_MANIFEST = (
    TEST_B_PATCH_DIR
    / "manifest.csv"
)

# --------------------------------------------------
# Output paths
# --------------------------------------------------
MODEL_DIR = PROJECT_ROOT / "Models"
BEST_MODEL_PATH = (
    MODEL_DIR
    / "SDENetFusion_Glas_best.pth"
)

LATEST_MODEL_PATH = (
    MODEL_DIR
    / "SDENetFusion_Glas_latest.pth"
)

TRAIN_HISTORY_PATH = (
    MODEL_DIR
    / "SDENetFusion_Glas_history.csv"
)

TEST_A_RESULTS_PATH = (
    MODEL_DIR
    / "SDENetFusion_Glas_test_a_results.csv"
)

TEST_B_RESULTS_PATH = (
    MODEL_DIR
    / "SDENetFusion_Glas_test_b_results.csv"
)

# --------------------------------------------------
# Training and patch settings
# --------------------------------------------------
PATCH_SIZE = 256
PATCH_STRIDE = 128

TRAIN_BATCH_SIZE = 8
TEST_BATCH_SIZE = 8

NUM_EPOCHS = 300
LEARNING_RATE = 1e-4

VALIDATION_SPLIT = 0.0
RANDOM_SEED = 42

NUM_SUPERPIXELS = 1000

In [ ]:
import re
import os
from glob import glob
from pathlib import Path

import torch
from torch import nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import tv_tensors
from torchvision.transforms import v2, InterpolationMode

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
import random
import sys

PROJECT_ROOT = Path("/data/notebooks/SDENetFusion")

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", os.getcwd())
print("Python path contains project root:", str(PROJECT_ROOT) in sys.path)

from SDENetFusion.Model import Model

Working directory: /content/SDENetFusion
Python path contains project root: True


In [7]:
def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_random_seed(RANDOM_SEED)

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [9]:
# def min_max_normalize(image):
#     image = image

#     minimum = image.amin()
#     maximum = image.amax()

#     image = (
#         image - minimum
#     ) / (
#         maximum - minimum
#     ).clamp_min(1e-6)

#     return tv_tensors.Image(image)

train_transforms = v2.Compose([
    # --------------------------------------------------
    # Basic geometric augmentation
    # --------------------------------------------------
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),

    v2.RandomChoice([
        v2.RandomRotation((0, 0)),
        v2.RandomRotation((90, 90)),
        v2.RandomRotation((180, 180)),
        v2.RandomRotation((270, 270)),
    ]),

    # --------------------------------------------------
    # Small translation, scaling, rotation, and shear
    # --------------------------------------------------
    v2.RandomApply(
        [
            v2.RandomAffine(
                degrees=10,
                translate=(0.05, 0.05),
                scale=(0.90, 1.10),
                shear=(-5, 5),
                interpolation=InterpolationMode.BILINEAR,
                fill={
                    tv_tensors.Image: 255,
                    tv_tensors.Mask: 0,
                },
            )
        ],
        p=0.30,
    ),

    # --------------------------------------------------
    # Mild tissue deformation
    # --------------------------------------------------
    v2.RandomApply(
        [
            v2.ElasticTransform(
                alpha=20.0,
                sigma=5.0,
                interpolation=InterpolationMode.BILINEAR,
                fill={
                    tv_tensors.Image: 255,
                    tv_tensors.Mask: 0,
                },
            )
        ],
        p=0.15,
    ),

    # --------------------------------------------------
    # Image appearance transforms
    # --------------------------------------------------
    v2.RandomApply([v2.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02,
    ),],
    p=0.30
    ),

    v2.RandomAdjustSharpness(
        sharpness_factor=1.5,
        p=0.20,
    ),

    v2.RandomAutocontrast(p=0.10),

    v2.RandomApply(
        [
            v2.GaussianBlur(
                kernel_size=3,
                sigma=(0.1, 1.0),
            )
        ],
        p=0.15,
    ),

    # --------------------------------------------------
    # Data type conversion
    # --------------------------------------------------
    v2.ToDtype(
        dtype={
            tv_tensors.Image: torch.float32,
            tv_tensors.Mask: torch.int64,
            "others": None,
        },
        scale=True,
    ),

])



In [10]:
evaluation_transforms = v2.Compose([
    v2.ToDtype(
        dtype={
            tv_tensors.Image: torch.float32,
            tv_tensors.Mask: torch.int64,
            "others": None,
        },
        scale=True,
    ),
])

validation_transforms = evaluation_transforms
test_transforms = evaluation_transforms

In [11]:
class GlasDataset(Dataset):

    def __init__(
        self,
        image_patch_dir,
        mask_patch_dir,
        image_names,
        transforms=None,
    ):
        self.image_patch_dir = Path(
            image_patch_dir
        )

        self.mask_patch_dir = Path(
            mask_patch_dir
        )

        self.transforms = transforms
        self.image_names = set(image_names)

        all_image_patch_paths = sorted(
            self.image_patch_dir.glob("*.npy")
        )

        self.samples = []

        for image_patch_path in all_image_patch_paths:
            patch_name = image_patch_path.stem

            source_image_name = (
                self.get_source_image_name(
                    patch_name
                )
            )

            if (
                source_image_name
                not in self.image_names
            ):
                continue

            mask_patch_path = (
                self.mask_patch_dir
                / image_patch_path.name
            )

            if not mask_patch_path.exists():
                raise FileNotFoundError(
                    f"Mask patch missing for "
                    f"{image_patch_path.name}: "
                    f"{mask_patch_path}"
                )

            self.samples.append({
                "image_path": image_patch_path,
                "mask_path": mask_patch_path,
                "patch_name": patch_name,
                "source_image_name": (
                    source_image_name
                ),
            })

        if not self.samples:
            raise ValueError(
                "No precomputed patches were found "
                "for the supplied image names."
            )

    @staticmethod
    def get_source_image_name(
        patch_name: str,
    ) -> str:
        match = re.match(
            r"^(.+)_patch_[0-9]+$",
            patch_name,
        )

        if match is None:
            raise ValueError(
                f"Patch name '{patch_name}' "
                f"must follow "
                f"'source_name_patch_0000'."
            )

        return match.group(1)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]

        image_numpy = np.load(
            sample["image_path"],
            allow_pickle=False,
        )

        mask_numpy = np.load(
            sample["mask_path"],
            allow_pickle=False,
        )

        image_tensor = (
            torch.from_numpy(image_numpy)
            .permute(2, 0, 1)
            .contiguous()
        )

        mask_tensor = torch.from_numpy(
            mask_numpy
        )

        image_tensor = tv_tensors.Image(
            image_tensor
        )

        mask_tensor = tv_tensors.Mask(
            mask_tensor
        )

        if self.transforms is not None:
            (
                image_tensor,
                mask_tensor,
            ) = self.transforms(
                image_tensor,
                mask_tensor,
            )

        mask_tensor = (
            mask_tensor > 0
        ).float()

        if mask_tensor.ndim == 2:
            mask_tensor = (
                mask_tensor.unsqueeze(0)
            )

        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "patch_name": (
                sample["patch_name"]
            ),
            "source_image_name": (
                sample["source_image_name"]
            ),
        }

In [12]:
import random


def split_image_names(
    image_names,
    validation_fraction=0.20,
    random_seed=42,
):
    """
    Randomly divides original image names into train and validation.

    All patches from one original image remain in the same split.
    """

    image_names = list(image_names)

    random_generator = random.Random(
        random_seed
    )

    random_generator.shuffle(
        image_names
    )

    number_of_images = len(image_names)

    number_of_validation_images = max(
        0,
        round(
            number_of_images
            * validation_fraction
        ),
    )

    validation_image_names = sorted(
        image_names[
            :number_of_validation_images
        ]
    )

    training_image_names = sorted(
        image_names[
            number_of_validation_images:
        ]
    )

    return (
        training_image_names,
        validation_image_names,
    )

In [13]:
def get_original_image_names(
    image_patch_dir,
):
    image_patch_dir = Path(image_patch_dir)

    patch_paths = sorted(
        image_patch_dir.glob("*.npy")
    )

    image_names = {
        GlasDataset.get_source_image_name(
            patch_path.stem
        )
        for patch_path in patch_paths
    }

    return sorted(image_names)

In [14]:
TRAIN_PATCH_IMAGE_DIR

PosixPath('/content/SDENetFusion/Datasets/Glas/precomputed/train_256_stride_128/images')

In [15]:
image_patch_dir = TRAIN_PATCH_IMAGE_DIR
mask_patch_dir = TRAIN_PATCH_MASK_DIR

all_image_names = get_original_image_names(
    image_patch_dir
)

(
    training_image_names,
    validation_image_names,
) = split_image_names(
    image_names=all_image_names,
    validation_fraction=VALIDATION_SPLIT,
    random_seed=RANDOM_SEED,
)

train_dataset = GlasDataset(
    image_patch_dir=image_patch_dir,
    mask_patch_dir=mask_patch_dir,
    image_names=training_image_names,
    transforms=train_transforms,
)

# validation_dataset = GlasDataset(
#     image_patch_dir=image_patch_dir,
#     mask_patch_dir=mask_patch_dir,
#     image_names=validation_image_names,
#     transforms=validation_transforms,
# )
validation_dataset = None

In [16]:
from torch.utils.data import DataLoader


train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    # persistent_workers=True,
    drop_last=False,
)


# val_loader = DataLoader(
#     dataset=validation_dataset,
#     batch_size=TRAIN_BATCH_SIZE,
#     shuffle=False,
#     num_workers=0,
#     pin_memory=torch.cuda.is_available(),
#     # persistent_workers=True,
#     drop_last=False,
# )
val_loader = None

In [17]:
train_image_set = set(
    training_image_names
)

validation_image_set = set(
    validation_image_names
)

overlap = (
    train_image_set
    & validation_image_set
)

print(
    "Number of original images:",
    len(all_image_names),
)

print(
    "Training images:",
    len(training_image_names),
)

print(
    "Validation images:",
    len(validation_image_names),
)

print(
    "Training patches:",
    len(train_dataset),
)

if validation_dataset is not None:
    print(
    "Validation patches:",
    len(validation_dataset),
)

print(
    "Image overlap:",
    overlap,
)

Number of original images: 85
Training images: 85
Validation images: 0
Training patches: 1968
Image overlap: set()


In [18]:
batch = next(iter(train_loader))

print("Images:", batch["image"].shape)
print("Masks:", batch["mask"].shape)
print("Patch names:", batch["patch_name"][:5])
print(
    "Source images:",
    batch["source_image_name"][:5],
)

Images: torch.Size([8, 3, 256, 256])
Masks: torch.Size([8, 1, 256, 256])
Patch names: ['train_40_patch_0016', 'train_23_patch_0003', 'train_5_patch_0019', 'train_7_patch_0010', 'train_81_patch_0018']
Source images: ['train_40', 'train_23', 'train_5', 'train_7', 'train_81']


In [19]:
def resume(
    model,
    optimizer,
    path,
    device="cpu",
    scheduler=None,
):
    checkpoint_data = torch.load(
        path,
        map_location=device,
    )

    model.load_state_dict(
        checkpoint_data["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint_data["optimizer_state_dict"]
    )

    if (
        scheduler is not None
        and "scheduler_state_dict" in checkpoint_data
    ):
        scheduler.load_state_dict(
            checkpoint_data["scheduler_state_dict"]
        )

    start_epoch = checkpoint_data.get(
        "epoch",
        0,
    ) + 1

    best_score = checkpoint_data.get(
        "best_score",
        None,
    )

    extra = checkpoint_data.get(
        "extra",
        None,
    )

    print(
        f"Checkpoint loaded from: {path}\n"
        f"Resuming from epoch: {start_epoch}"
    )

    return (
        model,
        optimizer,
        start_epoch,
        best_score,
        extra,
    )

def checkpoint(
    model,
    optimizer,
    epoch,
    filename,
    scheduler=None,
    best_score=None,
    best_epoch=None,
    extra=None,
):
    save_dict = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }

    if scheduler is not None:
        save_dict["scheduler_state_dict"] = (
            scheduler.state_dict()
        )

    if best_score is not None:
        save_dict["best_score"] = best_score

    if best_epoch is not None:
        save_dict["best_epoch"] = best_epoch

    if extra is not None:
        save_dict["extra"] = extra

    os.makedirs(
        os.path.dirname(filename),
        exist_ok=True,
    )

    torch.save(
        save_dict,
        filename,
    )

    print(
        f"Checkpoint saved at epoch {epoch}"
    )

import os

import pandas as pd


def save_train_history(
    history_path,
    epochs,
    train_losses,
    train_dice_scores,
    train_iou_scores,
    val_losses,
    val_dice_scores,
    val_iou_scores,
):
    """
    Saves training and validation metrics to a CSV file.

    Each list must contain one value per epoch.
    """

    history_directory = os.path.dirname(history_path)

    if history_directory:
        os.makedirs(
            history_directory,
            exist_ok=True,
        )

    history_df = pd.DataFrame({
        "epoch": epochs,
        "train_loss": train_losses,
        "train_dice": train_dice_scores,
        "train_iou": train_iou_scores,
        "val_loss": val_losses,
        "val_dice": val_dice_scores,
        "val_iou": val_iou_scores,
    })

    history_df.to_csv(
        history_path,
        index=False,
    )

In [20]:
import torch


# --------------------------------------------------
# Soft Dice
# Uses sigmoid probabilities without thresholding.
# Suitable for Dice loss.
# --------------------------------------------------
def soft_dice(
    logits: torch.Tensor,
    targets: torch.Tensor,
    smooth: float = 1e-6,
) -> torch.Tensor:
    probabilities = torch.sigmoid(logits)

    probabilities = probabilities.reshape(
        probabilities.shape[0],
        -1,
    )

    targets = targets.reshape(
        targets.shape[0],
        -1,
    )

    intersection = (
        probabilities * targets
    ).sum(dim=1)

    denominator = (
        probabilities.sum(dim=1)
        + targets.sum(dim=1)
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        denominator + smooth
    )

    return dice.mean()


# --------------------------------------------------
# Soft IoU
# Uses sigmoid probabilities without thresholding.
# --------------------------------------------------
def soft_iou(
    logits: torch.Tensor,
    targets: torch.Tensor,
    smooth: float = 1e-6,
) -> torch.Tensor:
    probabilities = torch.sigmoid(logits)

    probabilities = probabilities.reshape(
        probabilities.shape[0],
        -1,
    )

    targets = targets.reshape(
        targets.shape[0],
        -1,
    )

    intersection = (
        probabilities * targets
    ).sum(dim=1)

    union = (
        probabilities.sum(dim=1)
        + targets.sum(dim=1)
        - intersection
    )

    iou = (
        intersection + smooth
    ) / (
        union + smooth
    )

    return iou.mean()


# --------------------------------------------------
# Hard Dice
# Thresholds sigmoid probabilities at 0.5.
# Suitable for reporting train/validation/test metrics.
# --------------------------------------------------
def hard_dice(
    logits: torch.Tensor,
    targets: torch.Tensor,
    threshold: float = 0.5,
    smooth: float = 1e-6,
) -> torch.Tensor:
    predictions = (
        torch.sigmoid(logits) > threshold
    ).to(logits.dtype)

    predictions = predictions.reshape(
        predictions.shape[0],
        -1,
    )

    targets = targets.reshape(
        targets.shape[0],
        -1,
    )

    intersection = (
        predictions * targets
    ).sum(dim=1)

    denominator = (
        predictions.sum(dim=1)
        + targets.sum(dim=1)
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        denominator + smooth
    )

    return dice.mean()


# --------------------------------------------------
# Hard IoU
# Thresholds sigmoid probabilities at 0.5.
# Suitable for reporting train/validation/test metrics.
# --------------------------------------------------
def hard_iou(
    logits: torch.Tensor,
    targets: torch.Tensor,
    threshold: float = 0.5,
    smooth: float = 1e-6,
) -> torch.Tensor:
    predictions = (
        torch.sigmoid(logits) > threshold
    ).to(logits.dtype)

    predictions = predictions.reshape(
        predictions.shape[0],
        -1,
    )

    targets = targets.reshape(
        targets.shape[0],
        -1,
    )

    intersection = (
        predictions * targets
    ).sum(dim=1)

    union = (
        predictions.sum(dim=1)
        + targets.sum(dim=1)
        - intersection
    )

    iou = (
        intersection + smooth
    ) / (
        union + smooth
    )

    return iou.mean()


# --------------------------------------------------
# BCE + Dice loss
# --------------------------------------------------
def combined_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    bce_loss,
    alpha: float = 0.5,
    smooth: float = 1e-6,
) -> torch.Tensor:
    bce = bce_loss(
        logits,
        targets,
    )

    dice_loss = 1.0 - soft_dice(
        logits,
        targets,
        smooth=smooth,
    )

    return (
        alpha * bce
        + (1.0 - alpha) * dice_loss
    )

In [ ]:
def train(
    model,
    epoch,
    train_loader,
    val_loader,
    optimizer,
    device,
    BCE,
    dice_score=hard_dice,
    iou_score=hard_iou,
):
    #########################
    # Training
    #########################
    model.train()

    train_loss = 0.0
    train_dice = 0.0
    train_iou = 0.0

    for batch in tqdm(
        train_loader,
        desc=f"Epoch {epoch} [Train]",
    ):
        image = batch["image"].to(
            device,
            non_blocking=True,
        )

        mask = batch["mask"].to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad()

        logits = model(image)

        loss = (
            BCE(logits, mask)
            + (1 - soft_dice(logits, mask))
        )

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            dice = dice_score(logits, mask)
            iou = iou_score(logits, mask)

        train_loss += loss.item()
        train_dice += dice.item()
        train_iou += iou.item()

    train_loss /= len(train_loader)
    train_dice /= len(train_loader)
    train_iou /= len(train_loader)

    #########################
    # Validation
    #########################
    val_loss = 0.0
    val_dice = 0.0
    val_iou = 0.0

    if val_loader is not None:

        model.eval()



        with torch.no_grad():

            for batch in tqdm(
                val_loader,
                desc=f"Epoch {epoch} [Val]",
        ):
                image = batch["image"].to(
                    device,
                    non_blocking=True,
                )

                mask = batch["mask"].to(
                    device,
                    non_blocking=True,
                )

                logits = model(image)

                loss = (
                    BCE(logits, mask)
                    + (1 - soft_dice(logits, mask))
                )

                dice = dice_score(logits, mask)
                iou = iou_score(logits, mask)

                val_loss += loss.item()
                val_dice += dice.item()
                val_iou += iou.item()

        val_loss /= len(val_loader)
        val_dice /= len(val_loader)
        val_iou /= len(val_loader)

    #########################
    # Print
    #########################
    print(
        f"Epoch {epoch}\n"
        f"Train | Loss: {train_loss:.4f} | Dice: {train_dice:.4f} | IoU: {train_iou:.4f}"
    )
    if val_loader is not None:
        print(
            f"Val   | Loss: {val_loss:.4f} | Dice: {val_dice:.4f} | IoU: {val_iou:.4f}"
        )

    return {
        "train_loss": train_loss,
        "train_dice": train_dice,
        "train_iou": train_iou,
        "val_loss": val_loss,
        "val_dice": val_dice,
        "val_iou": val_iou,
    }

In [22]:
def test(
    model,
    dataloader,
    device,
    BCE,
    epoch,
    dice_score=hard_dice,
    iou_score=hard_iou,
    mode="Test",
):
    model.eval()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0

    with torch.inference_mode():

        for batch in tqdm(
            dataloader,
            desc=mode,
        ):
            images = batch["image"].to(
                device,
                non_blocking=True,
            )

            masks = batch["mask"].to(
                device,
                non_blocking=True,
            )

            logits = model(images)

            loss = (
                BCE(logits, masks)
                + (1 - soft_dice(logits, masks))
            )

            dice = dice_score(logits, masks)
            iou = iou_score(logits, masks)

            total_loss += loss.item()
            total_dice += dice.item()
            total_iou += iou.item()

    number_of_batches = len(dataloader)

    avg_loss = total_loss / number_of_batches
    avg_dice = total_dice / number_of_batches
    avg_iou = total_iou / number_of_batches

    print(
        f"{mode} Epoch {epoch} | "
        f"Loss: {avg_loss:.4f} | "
        f"Dice: {avg_dice:.4f} | "
        f"IoU: {avg_iou:.4f}"
    )

    return {
        "loss": avg_loss,
        "dice": avg_dice,
        "iou": avg_iou,
    }

In [23]:
!pip install torchinfo

In [24]:
import torchinfo

model = Model(in_channels=3, num_classes=1, num_segments=NUM_SUPERPIXELS, graph_feature_dim=128, num_heads=4).to(device)

torchinfo.summary(model, (1,3,256,256))

/content/SDENetFusion/SDENetFusion/Decoder/FusionBlock.py:405: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.image_self_attention = nn.TransformerEncoder(


Layer (type:depth-idx)                                            Output Shape              Param #
Model                                                             [1, 1, 256, 256]          --
├─ImageEncoder: 1-1                                               [1, 256, 32, 32]          --
│    └─SDEBlock: 2-1                                              [1, 26, 256, 256]         --
│    │    └─SDOperator: 3-1                                       [1, 26, 256, 256]         28
│    │    └─SEOperator: 3-2                                       [1, 26, 256, 256]         715
│    └─ModuleDict: 2-2                                            --                        --
│    │    └─EncoderBlock: 3-3                                     [1, 64, 256, 256]         52,096
│    │    └─EncoderBlock: 3-4                                     [1, 128, 128, 128]        221,696
│    │    └─EncoderBlock: 3-5                                     [1, 128, 64, 64]          295,424
│    │    └─EncoderBlock: 3-6 

In [25]:
import time

In [39]:
train_start_time = time.time()
avg_epoch_run_time = 0
train_epochs = []

train_losses = []
train_dice_scores = []
train_iou_scores = []

val_losses = []
val_dice_scores = []
val_iou_scores = []


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

loss_fn = nn.BCEWithLogitsLoss()

epochs = NUM_EPOCHS

start_epoch = 0
best_score = -1.0
best_epoch = -1

best_checkpoint_path = BEST_MODEL_PATH

latest_checkpoint_path = LATEST_MODEL_PATH

history_path = TRAIN_HISTORY_PATH


# --------------------------------------------------
# Resume from latest checkpoint first
# --------------------------------------------------
if os.path.exists(latest_checkpoint_path):
    checkpoint_data = torch.load(
        latest_checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(
        checkpoint_data["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint_data["optimizer_state_dict"]
    )

    start_epoch = (
        checkpoint_data["epoch"] + 1
    )

    best_score = checkpoint_data.get(
        "best_score",
        -1.0,
    )

    best_epoch = checkpoint_data.get(
        "best_epoch",
        -1,
    )

    print(
        f"Resuming from epoch {start_epoch}"
    )


# --------------------------------------------------
# Load history and keep only completed epochs
# before start_epoch
# --------------------------------------------------
if os.path.exists(history_path):
    history_df = pd.read_csv(history_path)

    history_df = history_df[
        history_df["epoch"] < start_epoch
    ].copy()

    train_epochs = history_df[
        "epoch"
    ].tolist()

    train_losses = history_df[
        "train_loss"
    ].tolist()

    train_dice_scores = history_df[
        "train_dice"
    ].tolist()

    train_iou_scores = history_df[
        "train_iou"
    ].tolist()

    val_losses = history_df[
        "val_loss"
    ].tolist()

    val_dice_scores = history_df[
        "val_dice"
    ].tolist()

    val_iou_scores = history_df[
        "val_iou"
    ].tolist()


# --------------------------------------------------
# Training loop
# --------------------------------------------------
for epoch in range(
    start_epoch,
    epochs,
):
    epoch_start_time = time.time()
    results = train(
        model=model,
        epoch=epoch,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        BCE=loss_fn,
    )
    epoch_end_time = time.time()

    train_loss = results["train_loss"]
    train_dice = results["train_dice"]
    train_iou = results["train_iou"]

    val_loss = results["val_loss"]
    val_dice = results["val_dice"]
    val_iou = results["val_iou"]

    train_epochs.append(epoch)

    train_losses.append(train_loss)
    train_dice_scores.append(train_dice)
    train_iou_scores.append(train_iou)

    val_losses.append(val_loss)
    val_dice_scores.append(val_dice)
    val_iou_scores.append(val_iou)

    save_train_history(
        history_path=history_path,
        epochs=train_epochs,
        train_losses=train_losses,
        train_dice_scores=train_dice_scores,
        train_iou_scores=train_iou_scores,
        val_losses=val_losses,
        val_dice_scores=val_dice_scores,
        val_iou_scores=val_iou_scores,
    )

    # Validation metrics determine the best model.
    if val_loader is not None:
        score = (
            val_dice + val_iou
        ) / 2.0
    else:
        score = (
            train_dice + train_iou
        ) / 2.0

    if score > best_score:
        best_score = score
        best_epoch = epoch

        checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            filename=best_checkpoint_path,
            best_score=best_score,
            best_epoch=best_epoch,
        )
        if val_loader is not None:
            print("Validation Set used....")
        print(
            f"New best model saved | "
            f"Epoch: {epoch} | "
            f"Score: {best_score:.4f}"
        )

    # Always save the most recent training state.
    checkpoint(
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        filename=latest_checkpoint_path,
        best_score=best_score,
        best_epoch=best_epoch,
    )

    print("%" * 100)
    avg_epoch_run_time += epoch_end_time - epoch_start_time


train_end_time = time.time()
print(
    f"Best epoch: {best_epoch} | "
    f"Best score: "
    f"{best_score:.4f}"
)
print(f"Total Training Time = {train_end_time - train_start_time}")
print(f"Average Epoch Run Time = {avg_epoch_run_time / NUM_EPOCHS}")


Resuming from epoch 95


Epoch 95 [Train]:   5%|▌         | 13/246 [00:14<04:23,  1.13s/it]


KeyboardInterrupt: 

In [35]:
from create_patches import (
    load_mask_numpy,
    generate_positions,
    pad_image_and_mask,
    preprocess_glaS_patches
)

In [27]:
def test_full_images(
    model,
    image_dir,
    annotation_dir,
    device,
    transforms,
    patch_size=256,
    stride=256,
    batch_size=8,
    threshold=0.5,
    smooth=1e-6,
):
    """
    Tests the model by rebuilding every original full-size prediction.

    For overlapping patches, sigmoid probabilities are averaged before
    thresholding.

    Returns:
        Dictionary containing mean full-image BCE, Dice and IoU.
    """
    model.eval()

    image_dir = Path(image_dir)
    annotation_dir = Path(annotation_dir)

    image_paths = sorted(
        image_dir.glob("*.bmp")
    )

    if not image_paths:
        raise FileNotFoundError(
            f"No BMP images found in {image_dir}"
        )

    image_results = []
    mask_predictions = []
    with torch.inference_mode():
        for image_path in tqdm(
            image_paths,
            desc="Testing full images",
        ):
            # --------------------------------------------------
            # 1. Read original full image and full mask
            # --------------------------------------------------
            image = np.asarray(
                Image.open(image_path).convert("RGB"),
                dtype=np.uint8,
            )

            annotation_path = (
                annotation_dir
                / f"{image_path.name}.json"
            )

            full_mask = load_mask_numpy(
                annotation_path
            )

            original_height, original_width = (
                image.shape[:2]
            )

            if full_mask.shape != (
                original_height,
                original_width,
            ):
                raise ValueError(
                    f"Image/mask size mismatch for "
                    f"{image_path.name}: "
                    f"image={image.shape[:2]}, "
                    f"mask={full_mask.shape}"
                )

            # --------------------------------------------------
            # 2. Pad images smaller than the patch size
            # --------------------------------------------------
            padded_image, padded_mask = (
                pad_image_and_mask(
                    image=image,
                    mask=full_mask,
                    patch_height=patch_size,
                    patch_width=patch_size
                )
            )

            padded_height, padded_width = (
                padded_image.shape[:2]
            )

            # --------------------------------------------------
            # 3. Recreate the patch positions
            # --------------------------------------------------
            top_positions = generate_positions(
                image_size=padded_height,
                patch_size=patch_size,
                stride=stride,
            )

            left_positions = generate_positions(
                image_size=padded_width,
                patch_size=patch_size,
                stride=stride,
            )

            patch_coordinates = [
                (top, left)
                for top in top_positions
                for left in left_positions
            ]

            # Probability sum and overlap count.
            probability_sum = torch.zeros(
                (padded_height, padded_width),
                dtype=torch.float32,
            )

            overlap_count = torch.zeros(
                (padded_height, padded_width),
                dtype=torch.float32,
            )

            # --------------------------------------------------
            # 4. Predict patches in mini-batches
            # --------------------------------------------------
            for start_index in range(
                0,
                len(patch_coordinates),
                batch_size,
            ):
                batch_coordinates = patch_coordinates[
                    start_index:start_index + batch_size
                ]

                patch_tensors = []

                for top, left in batch_coordinates:
                    image_patch = padded_image[
                        top:top + patch_size,
                        left:left + patch_size,
                        :,
                    ]

                    image_patch = (
                        torch.from_numpy(
                            image_patch.copy()
                        )
                        .permute(2, 0, 1)
                        .contiguous()
                    )

                    # A dummy mask is supplied because the transform
                    # pipeline expects an image-mask pair.
                    dummy_mask = torch.zeros(
                        (patch_size, patch_size),
                        dtype=torch.uint8,
                    )

                    image_patch = tv_tensors.Image(
                        image_patch
                    )

                    dummy_mask = tv_tensors.Mask(
                        dummy_mask
                    )

                    image_patch, _ = transforms(
                        image_patch,
                        dummy_mask,
                    )

                    patch_tensors.append(
                        image_patch
                    )

                image_batch = torch.stack(
                    patch_tensors,
                    dim=0,
                ).to(
                    device,
                    non_blocking=True,
                )

                logits = model(image_batch)

                # Expected model output:
                # [B, 1, patch_size, patch_size]
                if isinstance(logits, tuple):
                    logits = logits[0]

                probabilities = torch.sigmoid(
                    logits
                )

                if probabilities.ndim == 4:
                    probabilities = probabilities[:, 0]

                probabilities = (
                    probabilities
                    .detach()
                    .cpu()
                )

                # ----------------------------------------------
                # 5. Put predictions back into full image
                # ----------------------------------------------
                for patch_probability, (
                    top,
                    left,
                ) in zip(
                    probabilities,
                    batch_coordinates,
                ):
                    probability_sum[
                        top:top + patch_size,
                        left:left + patch_size,
                    ] += patch_probability

                    overlap_count[
                        top:top + patch_size,
                        left:left + patch_size,
                    ] += 1.0

            # --------------------------------------------------
            # 6. Average overlapping patch predictions
            # --------------------------------------------------
            reconstructed_probability = (
                probability_sum
                / overlap_count.clamp_min(1.0)
            )

            # Remove any padding added before patch generation.
            reconstructed_probability = (
                reconstructed_probability[
                    :original_height,
                    :original_width,
                ]
            )

            target = torch.from_numpy(
                full_mask
            ).float()

            prediction = (
                reconstructed_probability
                > threshold
            ).float()
            mask_predictions.append({
                "image_name": image_path.stem,
                "probability": (
                    reconstructed_probability.clone()
                ),
                "prediction": prediction.clone(),
                "target": target.clone(),
            })
            # --------------------------------------------------
            # 7. Calculate full-image metrics
            # --------------------------------------------------
            intersection = (
                prediction * target
            ).sum()

            prediction_sum = prediction.sum()
            target_sum = target.sum()

            dice = (
                2.0 * intersection + smooth
            ) / (
                prediction_sum
                + target_sum
                + smooth
            )

            union = (
                prediction_sum
                + target_sum
                - intersection
            )

            iou = (
                intersection + smooth
            ) / (
                union + smooth
            )

            # BCE calculated from reconstructed probabilities.
            bce = torch.nn.functional.binary_cross_entropy(
                reconstructed_probability.clamp(
                    min=1e-7,
                    max=1.0 - 1e-7,
                ),
                target,
            )

            image_results.append({
                "image_name": image_path.stem,
                "loss": bce.item(),
                "dice": dice.item(),
                "iou": iou.item(),
                "height": original_height,
                "width": original_width,
                "patch_count": len(
                    patch_coordinates
                ),
            })

            print(
                f"{image_path.stem} | "
                f"Size: {original_height}×{original_width} | "
                f"Patches: {len(patch_coordinates)} | "
                f"Dice: {dice.item():.4f} | "
                f"IoU: {iou.item():.4f}"
            )

    # ----------------------------------------------------------
    # 8. Dataset-level image averages
    # ----------------------------------------------------------
    mean_loss = float(
        np.mean([
            result["loss"]
            for result in image_results
        ])
    )

    mean_dice = float(
        np.mean([
            result["dice"]
            for result in image_results
        ])
    )

    mean_iou = float(
        np.mean([
            result["iou"]
            for result in image_results
        ])
    )

    print("\n" + "=" * 70)
    print(
        f"Test results over {len(image_results)} "
        f"original images"
    )
    print(f"Mean BCE:  {mean_loss:.4f}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Mean IoU:  {mean_iou:.4f}")
    print("=" * 70)

    return {
        "loss": mean_loss,
        "dice": mean_dice,
        "iou": mean_iou,
        "per_image": image_results,
        "mask_predictions": mask_predictions
    }

In [34]:
test_a_img_path = TEST_A_IMAGE_DIR
test_a_ann_path = TEST_A_ANNOTATION_DIR

test_b_img_path = TEST_B_IMAGE_DIR
test_b_ann_path = TEST_B_ANNOTATION_DIR

In [29]:
BEST_MODEL_PATH

PosixPath('/content/SDENetFusion/Models/SDENetFusion_Glas_best.pth')

In [30]:
best_checkpoint_path = BEST_MODEL_PATH

checkpoint_data = torch.load(
    best_checkpoint_path,
    map_location=device,
)

model.load_state_dict(
    checkpoint_data["model_state_dict"]
)

model = model.to(device)

In [31]:
test_a_results = test_full_images(
    model=model,
    image_dir=test_a_img_path,
    annotation_dir=test_a_ann_path,
    device=device,
    transforms=test_transforms,
    patch_size=256,
    stride=128,
    batch_size=8,
    threshold=0.5,
)

Testing full images:   2%|▏         | 1/60 [00:02<02:43,  2.77s/it]

testA_1 | Size: 522×775 | Patches: 24 | Dice: 0.9512 | IoU: 0.9069


Testing full images:   3%|▎         | 2/60 [00:05<02:35,  2.68s/it]

testA_10 | Size: 522×775 | Patches: 24 | Dice: 0.9042 | IoU: 0.8252


Testing full images:   5%|▌         | 3/60 [00:07<02:28,  2.61s/it]

...

testA_7 | Size: 522×775 | Patches: 24 | Dice: 0.8849 | IoU: 0.7936


Testing full images:  98%|█████████▊| 59/60 [02:22<00:02,  2.59s/it]

testA_8 | Size: 522×775 | Patches: 24 | Dice: 0.9453 | IoU: 0.8963


Testing full images: 100%|██████████| 60/60 [02:25<00:00,  2.42s/it]

testA_9 | Size: 522×775 | Patches: 24 | Dice: 0.9660 | IoU: 0.9343

Test results over 60 original images
Mean BCE:  0.2192
Mean Dice: 0.9281
Mean IoU:  0.8714


In [32]:
test_a_results_df = pd.DataFrame(
    test_a_results["per_image"]
)

test_a_results_df.to_csv(
    TEST_A_RESULTS_PATH,
    index=False,
)

In [33]:
TEST_A_RESULTS_PATH

PosixPath('/content/SDENetFusion/Models/SDENetFusion_Glas_test_a_results.csv')

In [ ]:
test_a_results = test_full_images(
    model=model,
    image_dir=test_b_img_path,
    annotation_dir=test_b_ann_path,
    device=device,
    transforms=test_transforms,
    patch_size=256,
    stride=128,
    batch_size=8,
    threshold=0.5,
)

In [50]:
preprocess_glaS_patches(
    image_dir=(
        r"/content/SDENetFusion/Datasets/Glas/test_b/img"
    ),
    annotation_dir=(
        r"/content/SDENetFusion/Datasets/Glas/test_b/ann"
    ),
    output_dir=(r"/content/SDENetFusion/Datasets/Glas/precomputed/test_b_256_stride_128"
    ),
    patch_size=256,
    stride=128,
)

testB_1.bmp: 24 patches
testB_10.bmp: 24 patches
testB_11.bmp: 24 patches
testB_12.bmp: 24 patches
testB_13.bmp: 24 patches
testB_14.bmp: 24 patches
testB_15.bmp: 24 patches
testB_16.bmp: 24 patches
testB_17.bmp: 24 patches
testB_18.bmp: 24 patches
testB_19.bmp: 24 patches
testB_2.bmp: 24 patches
testB_20.bmp: 24 patches
testB_3.bmp: 24 patches
testB_4.bmp: 24 patches
testB_5.bmp: 24 patches
testB_6.bmp: 24 patches
testB_7.bmp: 24 patches
testB_8.bmp: 24 patches
testB_9.bmp: 24 patches

Images processed: 20
Total patches: 480
Manifest: /content/SDENetFusion/Datasets/Glas/precomputed/test_b_256_stride_128/manifest.csv


In [51]:
test_b_results = test_full_images(
    model=model,
    image_dir=test_b_img_path,
    annotation_dir=test_b_ann_path,
    device=device,
    transforms=test_transforms,
    patch_size=256,
    stride=128,
    batch_size=8,
    threshold=0.5,
)

Testing full images:   5%|▌         | 1/20 [00:02<00:48,  2.58s/it]

testB_1 | Size: 522×775 | Patches: 24 | Dice: 0.8371 | IoU: 0.7199


Testing full images:  10%|█         | 2/20 [00:05<00:47,  2.61s/it]

testB_10 | Size: 522×775 | Patches: 24 | Dice: 0.9237 | IoU: 0.8582


Testing full images:  15%|█▌        | 3/20 [00:07<00:44,  2.60s/it]

...

testB_7 | Size: 522×775 | Patches: 24 | Dice: 0.9565 | IoU: 0.9167


Testing full images:  95%|█████████▌| 19/20 [00:49<00:02,  2.59s/it]

testB_8 | Size: 522×775 | Patches: 24 | Dice: 0.9572 | IoU: 0.9180


Testing full images: 100%|██████████| 20/20 [00:51<00:00,  2.59s/it]

testB_9 | Size: 522×775 | Patches: 24 | Dice: 0.7747 | IoU: 0.6323

Test results over 20 original images
Mean BCE:  0.3187
Mean Dice: 0.9042
Mean IoU:  0.8344


In [52]:
test_b_results_df = pd.DataFrame(
    test_b_results["per_image"]
)

test_b_results_df.to_csv(
    TEST_B_RESULTS_PATH,
    index=False,
)
#

In [55]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image


# Change this to your preferred output directory.
OUTPUT_DIR = Path("/content/SDENetFusion/Results/Test_A")

PROBABILITY_DIR = OUTPUT_DIR / "probability"
PREDICTION_DIR = OUTPUT_DIR / "prediction"
TARGET_DIR = OUTPUT_DIR / "target"

for directory in [
    PROBABILITY_DIR,
    PREDICTION_DIR,
    TARGET_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def tensor_to_numpy(tensor):
    """
    Converts a PyTorch tensor to a 2D NumPy array.

    Supports:
        [H, W]
        [1, H, W]
        [1, 1, H, W]
    """
    if isinstance(tensor, torch.Tensor):
        tensor = tensor.detach().cpu()

    array = np.asarray(tensor)
    array = np.squeeze(array)

    if array.ndim != 2:
        raise ValueError(
            f"Expected a 2D image after squeezing, "
            f"but received shape {array.shape}."
        )

    return array


for result in test_a_results["mask_predictions"]:
    image_name = result["image_name"]

    probability = tensor_to_numpy(
        result["probability"]
    )

    prediction = tensor_to_numpy(
        result["prediction"]
    )

    target = tensor_to_numpy(
        result["target"]
    )

    # Probability values [0, 1] -> grayscale values [0, 255].
    probability_image = np.clip(
        probability * 255.0,
        0,
        255,
    ).astype(np.uint8)

    # Binary masks -> black and white images.
    prediction_image = (
        prediction > 0.5
    ).astype(np.uint8) * 255

    target_image = (
        target > 0.5
    ).astype(np.uint8) * 255

    Image.fromarray(
        probability_image,
        mode="L",
    ).save(
        PROBABILITY_DIR
        / f"{image_name}_probability.png"
    )

    Image.fromarray(
        prediction_image,
        mode="L",
    ).save(
        PREDICTION_DIR
        / f"{image_name}_prediction.png"
    )

    Image.fromarray(
        target_image,
        mode="L",
    ).save(
        TARGET_DIR
        / f"{image_name}_target.png"
    )

print(f"Images saved to: {OUTPUT_DIR}")

/tmp/ipykernel_560/4102226649.py:78: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(
/tmp/ipykernel_560/4102226649.py:86: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(
/tmp/ipykernel_560/4102226649.py:94: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(


Images saved to: /content/SDENetFusion/Results/Test_A
